# Anatomy of AI Agents

The purpose of this talk is to demystify AI Agents. For this we would build Agents from very basic primitives avoiding usage of more complex frameworks
that obscure implementation.

- **AI Agent** an abstract entity that encapsulates a set of instructions with a set of tools it can use.
- **Multi-agent system** a set of AI Agents that interact with each other.

During this session we will:
- Dive deep into "anatomy" of programmatic LLM interaction
- Discuss Structural Output and Function (Tool) Calling
- Build [ReAct](https://arxiv.org/abs/2210.03629) agent **from scratch** 
- Create multiple agents that interact with each other
- Maybe look at some frameworks that implement multi-agent systems and streamline agentic workflows

## Used Libraries
- [langchain](https://python.langchain.com/docs/introduction/) is the most popular open-source library that provides both low-level and high-level primitives that simplify interactions with LLMs
- [pydantic](https://docs.pydantic.dev/latest/) is the most widely used data validation library for Python. We need it to define models for structural output.
- [LangFuse](https://langfuse.com/) observibility tool that can be run locally.

## Utilities
Just few useful functions

In [26]:
from IPython.display import Markdown, display
import json

def print_markdown(md_text):
    display(Markdown(md_text))


def print_message(obj):
    obj_dict = obj.__dict__
    obj_type = type(obj).__name__
    # Pretty print the dictionary as a JSON string
    pretty_json = json.dumps(obj_dict, indent=4)
    # Print the type and the formatted JSON string
    print(f"Type: {obj_type}")
    print(pretty_json)

## LLM
There are multiple LLMs we can use, some call through API and some run locally. For this example you can do both:
- **Local**:  using [ollama](https://ollama.com/). Please first visit the website and install the tool if you haven't already. After you are done, run:
```bash
ollama pull <model_name>
```
This will start the model on your local machine. Note this model is small and the results might be suboptimal. You can make it work, but it would take some time tuning prompts.

- **API**: using [OpenRouter API](https://openrouter.ai/)(or any other provider you prefer). You need to have an account and API key to use it. For OpenRouter, you need to set `OPENROUTER_API_KEY` environment variable. Using langchain, you can easely use most of the LLM providers, just swap the `llm` below.

Next we simply import a client for ollama from langchain.

In [54]:
import os
from dotenv import load_dotenv
from langfuse import Langfuse
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI
from otel import OTELCompliantLangfuseHandler

load_dotenv()

# This is an observability tool that can be run locally
langfuse_client = Langfuse()

callbacks = [OTELCompliantLangfuseHandler()]
use_ollama = True
if use_ollama:
    llm = ChatOllama(
        model="gemma4:26b",
        temperature=0,
        callbacks=callbacks
    )
else:
   llm = ChatOpenAI(
       model="openai/gpt-4.1-mini",  # OpenRouter model identifier
       openai_api_key=os.getenv('OPENROUTER_API_KEY'),
       openai_api_base="https://openrouter.ai/api/v1",
       temperature=0,
       callbacks=callbacks,
)

## Chatting with the model
Next we will run simple inference with the model.

Things to note here:
- We are sending the model messages. Each message has `role` and `content`. There are three types of roles:
    - `system` - contains a set of initial instructions for model to follow. This is the first message in the list. There could be only one system message
    - `human` - dah!
    - `ai` LLM response
    - `tool` result of running function (tool) requested by the LLM
- There could be as many `human`, `assistant` and `tool` messages as long as they fit into the context window.

In [55]:
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage, AIMessage

messages = [
    SystemMessage("You are a helpful assistant."),
    HumanMessage("What is the capital of France?"),
]
ai_msg = llm.invoke(messages)
print_message(ai_msg)

Type: AIMessage
{
    "content": "The capital of France is **Paris**.",
    "additional_kwargs": {},
    "response_metadata": {
        "model": "gemma4:26b",
        "created_at": "2026-05-22T19:01:06.762241Z",
        "done": true,
        "done_reason": "stop",
        "total_duration": 10450110500,
        "load_duration": 9321365667,
        "prompt_eval_count": 29,
        "prompt_eval_duration": 200657917,
        "eval_count": 37,
        "eval_duration": 784896295,
        "logprobs": null,
        "model_name": "gemma4:26b",
        "model_provider": "ollama"
    },
    "type": "ai",
    "name": null,
    "id": "lc_run--019e510f-e373-7622-ab08-46eb3326ac00-0",
    "tool_calls": [],
    "invalid_tool_calls": [],
    "usage_metadata": {
        "input_tokens": 29,
        "output_tokens": 37,
        "total_tokens": 66
    }
}


## Multi-message conversations
Since LLM is stateless, it is your responsibility to maintain the history of conversation. With every new message you need to keep sending all conversation to the model.

In [29]:
messages += [
    ai_msg,
    HumanMessage("What is it famous for?")
]
ai_msg = llm.invoke(messages)
print(ai_msg.content)


Paris is famous for many things, including:

1. **Landmarks:** Iconic sites such as the Eiffel Tower, Notre-Dame Cathedral, the Louvre Museum, and the Arc de Triomphe.
2. **Art and Culture:** Home to world-renowned museums like the Louvre and Musée d'Orsay, and a rich history of art, literature, and philosophy.
3. **Fashion:** Known as one of the fashion capitals of the world, hosting Paris Fashion Week and numerous high-end designer boutiques.
4. **Cuisine:** Famous for its gourmet food, including pastries like croissants and macarons, as well as fine dining and cafés.
5. **Romance:** Often called "The City of Love," it is a popular destination for couples and honeymooners.
6. **History:** A city with a deep historical background, from medieval times through the French Revolution to modern-day France.

These are just a few highlights that make Paris a globally renowned city.


In [30]:
print_markdown(ai_msg.content)

Paris is famous for many things, including:

1. **Landmarks:** Iconic sites such as the Eiffel Tower, Notre-Dame Cathedral, the Louvre Museum, and the Arc de Triomphe.
2. **Art and Culture:** Home to world-renowned museums like the Louvre and Musée d'Orsay, and a rich history of art, literature, and philosophy.
3. **Fashion:** Known as one of the fashion capitals of the world, hosting Paris Fashion Week and numerous high-end designer boutiques.
4. **Cuisine:** Famous for its gourmet food, including pastries like croissants and macarons, as well as fine dining and cafés.
5. **Romance:** Often called "The City of Love," it is a popular destination for couples and honeymooners.
6. **History:** A city with a deep historical background, from medieval times through the French Revolution to modern-day France.

These are just a few highlights that make Paris a globally renowned city.

## Structural output
Dealing with unstructured text programmatically is hard and unreliable. So let's ask the model to respond in JSON format.
  

In [31]:
messages = [
    SystemMessage("Your goal is to identify the topic of the question and answer it in JSON format."
                  " The output should contain `reply` and `topic` fields."
                  " Return only JSON, do not wrap the response."),
    HumanMessage("What is the capital of France?"),

]
ai_msg = llm.invoke(messages)
print(ai_msg.content)

{
  "topic": "Geography",
  "reply": "The capital of France is Paris."
}


### Structural output via langchain instrumentation 
If we only have one type of output we can simply declare a pydantic model and ask langchain to output using that model

In [32]:
from pydantic import BaseModel, Field


class AnswerWithTopic(BaseModel):
    reply: str = Field(description="Reply to the question")
    topic: str = Field(description="General topic of the questions")


structured_llm = llm.with_structured_output(AnswerWithTopic)
result = structured_llm.invoke("What is the capital of France?")
result

AnswerWithTopic(reply='The capital of France is Paris.', topic='Geography')

## Structural output via Function Calling
Some LLMs support [function/tool calling](https://docs.langchain.com/oss/python/langchain/tools#tools). Simply put, along with the message model returns a function call, given you have provided the definition of the function in your request.

As a matter of fact, previous example (`llm.with_structured_output(AnswerWithTopic)`) does this already, LangChain simply uses function calling "under the hood". Here is how function definition produced by LangChain actually looks like:

```json
 {
  "type": "function",
  "function": {
    "name": "AnswerWithTopic",
    "description": "",
    "parameters": {
      "properties": {
        "reply": {
          "description": "Reply to the question",
          "type": "string"
        },
        "topic": {
          "description": "General topic of the questions",
          "type": "string"
        }
      },
      "required": [
        "reply",
        "topic"
      ],
      "type": "object"
    }
  }
}
```

There are multiple ways to configure functions with an LLM, and there are a lot of LLMs, this is why we are using LangChain to abstract all that for us.

Another way to configure a function with LangChain is to decorate a python function with `@tool`.

In [33]:
from langchain_core.tools import tool


@tool
def multiply(a: int, b: int) -> int:
    """
    Multiply two numbers.
    """
    return a * b


print('type:', type(multiply))
print('Call tool directly:', multiply.invoke({"a": 2, "b": 3}))
print('name:', multiply.name)
print('description:', multiply.description)
print('args:', multiply.args)

type: <class 'langchain_core.tools.structured.StructuredTool'>
Call tool directly: 6
name: multiply
description: Multiply two numbers.
args: {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


### Tool Usage Example
In this cell, we demonstrate how to use a tool within the LangChain framework. 
We define a simple tool called `multiply` that multiplies two numbers. 
We then bind this tool to the language model and invoke it with a query. 
The language model identifies the need to use the tool, and tells us to use the tool with specific parameters.

In [34]:
tools = [multiply]
llm_with_tools = llm.bind_tools(tools)

ai_msg = llm_with_tools.invoke("What is 42 * 177?")
print_message(ai_msg)

Type: AIMessage
{
    "content": "",
    "additional_kwargs": {
        "refusal": null
    },
    "response_metadata": {
        "token_usage": {
            "completion_tokens": 18,
            "prompt_tokens": 46,
            "total_tokens": 64,
            "completion_tokens_details": {
                "accepted_prediction_tokens": null,
                "audio_tokens": 0,
                "reasoning_tokens": 0,
                "rejected_prediction_tokens": null,
                "image_tokens": 0
            },
            "prompt_tokens_details": {
                "audio_tokens": 0,
                "cached_tokens": 0,
                "cache_write_tokens": 0,
                "video_tokens": 0
            },
            "cost": 4.72e-05,
            "is_byok": false,
            "cost_details": {
                "upstream_inference_cost": 4.72e-05,
                "upstream_inference_prompt_cost": 1.84e-05,
                "upstream_inference_completions_cost": 2.88e-05
            }


In [35]:
print(ai_msg.tool_calls)

[{'name': 'multiply', 'args': {'a': 42, 'b': 177}, 'id': 'call_dUG9Qy9IM9eH9ubmmRSP5yo2', 'type': 'tool_call'}]


### invoke_tools Function

The `invoke_tools` function is used to call and execute various tools within our agent system. It ensures that tools are invoked correctly and handles any errors that may occur during execution. We will make heavy use of this function with agents.

In [36]:
from langchain_core.tools import StructuredTool
from langchain_core.messages import ToolCall
from typing import List
from langfuse import observe
import json


def invoke_tools(tool_calls: list[ToolCall], tools: list[StructuredTool]) -> List[ToolMessage]:
    """
    Invokes a list of tool calls using the provided tools and returns the results.
    Args:
        tool_calls (list[ToolCall]): A list of tool call dictionaries, each containing the name of the tool to invoke and other necessary parameters.
            An LLM produces this with AIMessage.
        tools (list[StructuredTool]): A list of StructuredTool objects available for invocation. Those are @tool decorated functions in our case.
    Returns:
        List[ToolMessage]: A list of ToolMessage objects containing the results of the tool invocations. 
            This is the message we will be putting into the context so that the LLM "knows" what results the function call produced.
    Raises:
        ValueError: If any item in the tools list is not an instance of StructuredTool.
    """
    for tool in tools:
        if type(tool) != StructuredTool:
            raise ValueError("Tool", tool, " is not a StructuredTool")
    results = []
    tools_dict = {tool.name: tool for tool in tools}
    for call in tool_calls:
        print('🔧---> Invoking tool call', call)
        tool_name = call['name']
        tool = tools_dict.get(tool_name)
        if tool:
            with langfuse_client.start_as_current_observation(name=f"🛠️ {tool_name}", as_type="tool") as tool_obs:
                tool_obs.update(input=call)
                tool_result = tool.invoke(call)  # Raw result from a tool
                tool_obs.update(output=tool_result)

                # Convert result to string for ToolMessage content
                if isinstance(tool_result, str):
                    content = tool_result
                else:
                    # Convert complex objects to JSON string
                    content = json.dumps(tool_result, default=str)

                # Create ToolMessage with string content
                result = ToolMessage(
                    name=tool_name,
                    content=content,
                    tool_call_id=call['id']
                )
        else:
            result = ToolMessage(
                content=f"Error: Tool with name `{tool_name}` is not available. Use only the available tools.",
                tool_call_id=call['id']
            )
        print('🔧<--- Tool call result:', result)
        results.append(result)
    return results

In [37]:
calls = ai_msg.tool_calls
tools = [multiply]
invoke_tools(calls, tools)

🔧---> Invoking tool call {'name': 'multiply', 'args': {'a': 42, 'b': 177}, 'id': 'call_dUG9Qy9IM9eH9ubmmRSP5yo2', 'type': 'tool_call'}
🔧<--- Tool call result: content='"content=\'7434\' name=\'multiply\' tool_call_id=\'call_dUG9Qy9IM9eH9ubmmRSP5yo2\'"' name='multiply' tool_call_id='call_dUG9Qy9IM9eH9ubmmRSP5yo2'


[ToolMessage(content='"content=\'7434\' name=\'multiply\' tool_call_id=\'call_dUG9Qy9IM9eH9ubmmRSP5yo2\'"', name='multiply', tool_call_id='call_dUG9Qy9IM9eH9ubmmRSP5yo2')]

# AI Agent
Let's define a simple AI agent

There is nothing fundamentally different from what you already seen. Agent will simply encapsulate all the behaviour you have just seen:
1. It contains an LLM.
2. It has a *name*, a *persona* and optionally a *task*.
3. It has *tools* it can use, an ability to call the tools and observe the results.
4. It maintains the history of conversation and tools execution.

In [38]:
from langchain_core.language_models import BaseChatModel
from typing import Optional

class AgentReply(BaseModel):
    reply: str
    tool_call_results: Optional[List[ToolMessage]] = None


class ReActAgent(object):
    def __init__(self,
                 agent_llm: BaseChatModel,
                 name: str,
                 persona: str,
                 task: Optional[str] = None,
                 tools: Optional[List[StructuredTool]]=None,
                 ):
        """
       Initialize a ReActAgent.

       Parameters
       ----------
       agent_llm : BaseChatModel
           The language model to be used by the agent.
       name : str
           The name of the agent.
       persona : str
           The persona or role of the agent.
       task : Optional[str], optional
           The specific task assigned to the agent, by default None.
       tools : List[StructuredTool], optional
           A list of tools that the agent can use, by default an empty list.
       """
        if tools is None:
            tools = []
        self.name = name
        self.llm = agent_llm
        self.persona = persona
        self.task = task
        self.tools = tools
        self.messages = [
            SystemMessage(f"""{persona}
{'Your task is: ' + task if task else ''}
""")
        ]
        if tools:
            self.llm = self.llm.bind_tools(self.tools)

    @observe(as_type="agent")
    def call(self, message: str) -> AgentReply:
        """
        Calls the agent with a given message and processes the response.
        If the agent requests a tool call, it will be executed and the tool response will be sent back to the LLM.
        Args:
            message (str): The message to send to the agent.
        Returns:
            AgentReply: The response from the agent after processing the message.
        """
        langfuse_client.update_current_span(
            name=f"🤖💬 {self.name}",
            metadata={
                "agent": {
                    "agent_name": self.name,
                    "persona": self.persona,
                    "task": self.task,
                    "num_tools": len(self.tools)
                }
            }
        )
        print(f'💬---> Calling agent "{self.name}", with message:"{message}')

        self.messages.append(HumanMessage(message))
        response = self._call_llm_and_process_response()
        while self.is_last_message_tool_call():
            response = self._call_with_tool_result()
        return response

    def _call_with_tool_result(self) -> AgentReply:
        """
        Calls the agent with the tool result and manages the tool call.

        After an LLM requests a function call and the agent runs the functions with the given parameters, the agent
        needs to return the results to the LLM. The LLM calls this function to get the results of the function
        call and manage the tool call.

        Returns:
            AgentReply: An object containing the reply content and the tool call results.
        """
        print('💬🔧---> Calling agent', self.name, 'with tool result')
        reply = self._invoke_llm(self.messages)
        self.messages.append(reply)
        tool_call_results = self._manage_tool_call(reply.tool_calls)
        return AgentReply(reply=reply.content, tool_call_results=tool_call_results)

    def is_last_message_tool_call(self) -> bool:
        return isinstance(self.messages[-1], ToolMessage)

    def _manage_tool_call(self, tool_calls: list[ToolCall]) -> Optional[List[ToolMessage]]:
        if not tool_calls:
            return None
        tool_call_results = invoke_tools(tool_calls, self.tools)
        self.messages += tool_call_results
        return tool_call_results

    def _invoke_llm(self, messages: list):
        return self.llm.invoke(messages)

    def _call_llm_and_process_response(self):
        reply = self._invoke_llm(self.messages)
        self.messages.append(reply)
        tool_call_results = self._manage_tool_call(reply.tool_calls)
        agent_reply = AgentReply(reply=reply.content, tool_call_results=tool_call_results)
        if agent_reply.reply:
            print(f'💬<--- Agent {self.name} replied:', agent_reply.reply)
        return agent_reply


### Math agent

Now let's define a simple agent that will do math for us. First we will define a few tools for the agent, and create the agent using the class above

In [40]:
@tool
def add(a: int, b: int) -> int:
    """
    Add two numbers.
    """
    return a + b


@tool
def subtract(a: int, b: int) -> int:
    """
    Subtract two numbers.
    """
    return a - b


@tool
def divide(a: int, b: int) -> float:
    """
    Divide two numbers.
    """
    return a / b


agent = ReActAgent(llm,
                   name='Math Guru',
                   persona='You are an expert in mathematics',
                   task='Help the user to solve mathematical problems. '
                         'You will call math operations when needed, otherwise just reply to the user.'
                         'IMPORTANT: If you are asked to perform any operation that you do not have specific a tool for, reply "I don\'t know how to do this", do not say anything else!',
                   tools=[multiply, add, subtract, divide]
                   )

Now we can communicate with the agent and ask it to do math for us.

In [41]:
agent.call('What is 3 * 12?')

💬---> Calling agent "Math Guru", with message:"What is 3 * 12?
🔧---> Invoking tool call {'name': 'multiply', 'args': {'a': 3, 'b': 12}, 'id': 'call_iEZSzxYYjszH6WoEocTdfVBC', 'type': 'tool_call'}
🔧<--- Tool call result: content='"content=\'36\' name=\'multiply\' tool_call_id=\'call_iEZSzxYYjszH6WoEocTdfVBC\'"' name='multiply' tool_call_id='call_iEZSzxYYjszH6WoEocTdfVBC'
💬🔧---> Calling agent Math Guru with tool result


AgentReply(reply='3 * 12 is 36.', tool_call_results=None)

Since the agent maintains the state of the conversation we can ask it followup questions.

In [42]:
agent.call('What is the half of that?')

💬---> Calling agent "Math Guru", with message:"What is the half of that?
🔧---> Invoking tool call {'name': 'divide', 'args': {'a': 36, 'b': 2}, 'id': 'call_5teHbNPa7Jd9YVYtl8QxOCtW', 'type': 'tool_call'}
🔧<--- Tool call result: content='"content=\'18.0\' name=\'divide\' tool_call_id=\'call_5teHbNPa7Jd9YVYtl8QxOCtW\'"' name='divide' tool_call_id='call_5teHbNPa7Jd9YVYtl8QxOCtW'
💬🔧---> Calling agent Math Guru with tool result


AgentReply(reply='Half of 36 is 18.', tool_call_results=None)

As we gave the agent instructions to only do math operations it has tools for, when we ask it to do something else it should refuse.

If you are running this code with a local model, it most likelly will try to call the `factorial` function, which shows you the difference in "cognition" between models of different size.

In [43]:
agent.call('Calculate factorial of that number')

💬---> Calling agent "Math Guru", with message:"Calculate factorial of that number
💬<--- Agent Math Guru replied: I don't know how to do this


AgentReply(reply="I don't know how to do this", tool_call_results=None)

# Coding agent
Now let's make the agent a bit more complex. Let's create a coding agent that will write code for us.

In [44]:
from pathlib import Path
from typing import Optional, List

# Base directory - all file operations are scoped to this directory
BASE_DIR = Path("demo/code_agent/")

def _get_safe_path(relative_path: str) -> Path:
    """
    Convert a relative path to an absolute path within BASE_DIR.
    Prevents directory traversal attacks.
    """
    full_path = (BASE_DIR / relative_path).resolve()

    # Ensure it's within BASE_DIR
    try:
        full_path.relative_to(BASE_DIR.resolve())
    except ValueError:
        raise ValueError(f"Path {relative_path} is outside the allowed directory")

    return full_path

@tool
def list_files(path: Optional[str] = None) -> List[str]:
    """
    List all files and directories in the specified path.
    If no path is provided, lists the root directory.

    Parameters
    ----------
    path : Optional[str]
        Relative path within the agent's workspace.
        If None, lists the root directory.

    Returns
    -------
    List[str]
        List of file and directory names. Directories end with "/"
    """
    target_path = _get_safe_path(path or "")
    target_path.mkdir(parents=True, exist_ok=True)

    items = []
    for item in sorted(target_path.iterdir()):
        if item.is_dir():
            items.append(f"{item.name}/")
        else:
            items.append(item.name)

    return items

@tool
def read_file(path: str) -> str:
    """
    Read and return the contents of a file.

    Parameters
    ----------
    path : str
        Relative path to the file within the agent's workspace.

    Returns
    -------
    str
        The complete contents of the file
    """
    file_path = _get_safe_path(path)

    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    if not file_path.is_file():
        raise ValueError(f"Path is not a file: {path}")

    return file_path.read_text(encoding='utf-8')

@tool
def create_file(path: str, content: str) -> str:
    """
    Create a new file with the specified content.
    If the file already exists, this operation will fail.

    Parameters
    ----------
    path : str
        Relative path for the new file.
    content : str
        The content to write to the file

    Returns
    -------
    str
        Success message with the file path
    """
    file_path = _get_safe_path(path)

    if file_path.exists():
        raise FileExistsError(
            f"File already exists: {path}. Use edit_file to modify existing files."
        )

    file_path.parent.mkdir(parents=True, exist_ok=True)
    file_path.write_text(content, encoding='utf-8')

    return f"Successfully created file: {path} ({len(content)} characters)"

@tool
def edit_file(path: str, old_content: str, new_content: str) -> str:
    """
    Edit an existing file by replacing old_content with new_content.

    Parameters
    ----------
    path : str
        Relative path to the file to edit.
    old_content : str
        The exact text to find and replace.
        Use an empty string "" to append to the end of the file.
    new_content : str
        The text to replace old_content with.
        Use an empty string "" to delete old_content.

    Returns
    -------
    str
        Success message with details about the edit
    """
    file_path = _get_safe_path(path)

    if not file_path.exists():
        raise FileNotFoundError(
            f"File not found: {path}. Use create_file to create new files."
        )

    current_content = file_path.read_text(encoding='utf-8')

    # Handle append case
    if old_content == "":
        new_file_content = current_content + new_content
        file_path.write_text(new_file_content, encoding='utf-8')
        return f"Successfully edited file: {path} (appended {len(new_content)} characters)"

    # Handle delete case
    if new_content == "":
        if old_content not in current_content:
            raise ValueError(f"Content to delete not found in {path}")
        new_file_content = current_content.replace(old_content, "", 1)
        file_path.write_text(new_file_content, encoding='utf-8')
        return f"Successfully edited file: {path} (deleted {len(old_content)} characters)"

    # Handle replace case
    if old_content not in current_content:
        raise ValueError(
            f"Content to replace not found in {path}. "
            f"Make sure old_content matches exactly (including whitespace)."
        )

    occurrences = current_content.count(old_content)
    new_file_content = current_content.replace(old_content, new_content, 1)
    file_path.write_text(new_file_content, encoding='utf-8')

    return (
        f"Successfully edited file: {path} "
        f"(replaced 1 of {occurrences} occurrence(s))"
    )

@tool
def delete_file(path: str) -> str:
    """
    Delete a file.

    Parameters
    ----------
    path : str
        Relative path to the file to delete.

    Returns
    -------
    str
        Success message
    """
    file_path = _get_safe_path(path)

    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    if not file_path.is_file():
        raise ValueError(f"Path is not a file: {path}")

    file_path.unlink()

    return f"Successfully deleted file: {path}"

# Export all tools as a list
ALL_FILE_TOOLS = [list_files, read_file, create_file, edit_file, delete_file]

In [45]:
CODING_AGENT_TASK = """## Your Role
Help users create, read, edit, and manage code files. Write clean, well-structured code with proper documentation.

## Available Tools

1. **list_files(path)** - List files in a directory
   - Use to see what files exist before creating/editing
   - Call with no arguments to list root directory

2. **read_file(path)** - Read file contents
   - Use before editing to understand current content
   - Required to see what you're working with

3. **create_file(path, content)** - Create new files
   - Only for NEW files (fails if file exists)
   - Always include complete, working code
   - Add proper HTML structure, CSS styling, and JavaScript functionality

4. **edit_file(path, old_content, new_content)** - Modify existing files
   - old_content must match EXACTLY (including whitespace)
   - Use empty string "" for old_content to append to end of file
   - Use empty string "" for new_content to delete old_content
   - Read the file first to get exact text to replace

5. **delete_file(path)** - Delete files
   - Use sparingly, only when explicitly requested

## Workflow

1. **Understand the request** - What does the user want?
2. **Check existing files** - Use list_files() and read_file() to see what exists
3. **Plan your approach** - Decide whether to create new files or edit existing ones
4. **Execute** - Use the appropriate tools
5. **Verify** - Read back the file to confirm changes

## Best Practices

- Always read files before editing them
- When editing, copy the exact text (with whitespace) for old_content
- Write complete, functional code - no placeholders or TODOs
- Include proper error handling and edge cases
- Add comments to explain complex logic
- For HTML: include proper DOCTYPE, meta tags, and semantic structure
- For JavaScript: use modern ES6+ syntax
- For CSS: use clean, maintainable styles

## Response Style

- Be concise but clear
- Explain what you're doing and why
- Show the user what you created/changed
- If something fails, explain the error and try a different approach

## Example Interaction

User: "Create a simple clicker game"
You:
1. Call list_files() to check what exists
2. Call create_file("game.html", <complete HTML with game logic>)
3. Explain what you created

User: "Add a timer that shows how long they've been playing"
You:
1. Call read_file("game.html") to see current code
2. Call edit_file() to add timer functionality
3. Explain the changes made

Remember: You're a professional developer. Write production-quality code, not demos or sketches."""
coding_agent = ReActAgent(llm,
                          name='Coding Agent',
                          persona='You are an expert software developer assistant with access to file manipulation tools.',
                          task=CODING_AGENT_TASK,
                          tools=ALL_FILE_TOOLS
                          )

In [46]:
coding_agent.call("Create a simple HTML clicker game called game.html with a button that increments a score")

💬---> Calling agent "Coding Agent", with message:"Create a simple HTML clicker game called game.html with a button that increments a score
🔧---> Invoking tool call {'name': 'list_files', 'args': {'path': None}, 'id': 'call_dsPlEqnwe7lLqB44Mq2DB4r0', 'type': 'tool_call'}
🔧<--- Tool call result: content='"content=[] name=\'list_files\' tool_call_id=\'call_dsPlEqnwe7lLqB44Mq2DB4r0\'"' name='list_files' tool_call_id='call_dsPlEqnwe7lLqB44Mq2DB4r0'
💬🔧---> Calling agent Coding Agent with tool result
🔧---> Invoking tool call {'name': 'create_file', 'args': {'path': 'game.html', 'content': '<!DOCTYPE html>\n<html lang="en">\n<head>\n    <meta charset="UTF-8">\n    <meta name="viewport" content="width=device-width, initial-scale=1.0">\n    <title>Simple Clicker Game</title>\n    <style>\n        body {\n            font-family: Arial, sans-serif;\n            display: flex;\n            flex-direction: column;\n            align-items: center;\n            justify-content: center;\n            

AgentReply(reply='I have created a simple HTML clicker game in a file named game.html. It features a button that increments the score displayed on the screen each time it is clicked. The page includes basic styling for a clean and centered layout. You can open this file in a browser to play the game. Let me know if you want to add more features or make any changes.', tool_call_results=None)

In [47]:
coding_agent.call("Add an auto-clicker feature to game.html that increments the score every second after 5 seconds of play")

💬---> Calling agent "Coding Agent", with message:"Add an auto-clicker feature to game.html that increments the score every second after 5 seconds of play
🔧---> Invoking tool call {'name': 'read_file', 'args': {'path': 'game.html'}, 'id': 'call_k1cK8EWRRIz8IiGYNYYeN2l5', 'type': 'tool_call'}
🔧<--- Tool call result: content='"content=\'<!DOCTYPE html>\\\\n<html lang=\\"en\\">\\\\n<head>\\\\n    <meta charset=\\"UTF-8\\">\\\\n    <meta name=\\"viewport\\" content=\\"width=device-width, initial-scale=1.0\\">\\\\n    <title>Simple Clicker Game</title>\\\\n    <style>\\\\n        body {\\\\n            font-family: Arial, sans-serif;\\\\n            display: flex;\\\\n            flex-direction: column;\\\\n            align-items: center;\\\\n            justify-content: center;\\\\n            height: 100vh;\\\\n            margin: 0;\\\\n            background-color: #f0f0f0;\\\\n        }\\\\n        #score {\\\\n            font-size: 2rem;\\\\n            margin-bottom: 20px;\\\\n     

AgentReply(reply='I have added an auto-clicker feature to game.html. Now, after 5 seconds of play, the score will automatically increment by 1 every second in addition to manual clicks. This is implemented using a setTimeout to start a setInterval that updates the score every second. You can test it by opening the file in a browser and watching the score increase automatically after 5 seconds. Let me know if you want to customize the timing or add more features.', tool_call_results=None)

In [48]:
coding_agent.call("Change game.html to use a dark theme with neon green and purple colors")

💬---> Calling agent "Coding Agent", with message:"Change game.html to use a dark theme with neon green and purple colors
🔧---> Invoking tool call {'name': 'read_file', 'args': {'path': 'game.html'}, 'id': 'call_kKfFSgpYlRm6vY22dha48kXg', 'type': 'tool_call'}
🔧<--- Tool call result: content='"content=\'<!DOCTYPE html>\\\\n<html lang=\\"en\\">\\\\n<head>\\\\n    <meta charset=\\"UTF-8\\">\\\\n    <meta name=\\"viewport\\" content=\\"width=device-width, initial-scale=1.0\\">\\\\n    <title>Simple Clicker Game</title>\\\\n    <style>\\\\n        body {\\\\n            font-family: Arial, sans-serif;\\\\n            display: flex;\\\\n            flex-direction: column;\\\\n            align-items: center;\\\\n            justify-content: center;\\\\n            height: 100vh;\\\\n            margin: 0;\\\\n            background-color: #f0f0f0;\\\\n        }\\\\n        #score {\\\\n            font-size: 2rem;\\\\n            margin-bottom: 20px;\\\\n        }\\\\n        button {\\\\n   

AgentReply(reply='I have updated game.html to use a dark theme with neon green and purple colors. The background is now dark, the score text is neon green with a glow effect, and the button has a neon purple border and glow that changes to neon green on hover. This gives the game a vibrant, futuristic look. Open the file in a browser to see the new style in action. Let me know if you want any other style or feature changes.', tool_call_results=None)

# Multiple agents

The true power of AI Agents comes when agents are able to "talk" to each other. It is like when you "converse" with ChatGPT, when you send one message, even when your Propmting Skills are great, you will, in most cases get more out of it if you ask followups. With multi-agent, you replace yourself with another agent.

This can be done in multiple ways but we will make it so that agents can delegate execution to another agent. This will allow for more complex interactions between agents.

For this purpose we need to define a few new entities:
- **AgentGroup** - this class will orchestrate the interactions between agents and manage the conversation history.
- **Manager Agent** - this Agent will be responsible for receiving messages from the user and passing them to the agents.
- **delegate_task** tool - will allow passing execution to other agents.

Here is a conceptual graph of the interaction between the user, the receptionist, and the agents:
```mermaid
flowchart LR
    user[User]
    subgraph AgentGroup
        r[Manager]
        A[Agent 1]
        B[Agent 2]
    end
    user -->|0 message| AgentGroup
    r -->|2 message| A
    A -->|3 Delegate|B
    B -->|4 Delegate|r
    r -->|5 Delegate|B
    r -->|6 Reply| user
```
Agents would delegate to each other until the conversation is resolved.

In [49]:
from typing import Union


class HandOff(BaseModel):
    agent: str
    message: str
    summary: str


# Signals that the group of agents should stop processing the message
STOP = "STOP"
HAND_OFF_TOOL = "delegate_task"


def create_handoff_tool(delegate_task_stack: List[HandOff]):
    """
    Create a tool that allows an agent to hand off a task to another agent. 
    Because we need to keep track of the handoff information, we pass in a stack to store the handoff information.

    Parameters:
        delegate_task_stack: The stack to store the handoff information.
    """

    @tool
    def delegate_task(agent_name: str, message: str, summary: str) -> Union[HandOff, str]:
        """
        Hand off the conversation to another agent.
        Always call this tool when a more specialized agent is available to complete the task.

        Parameters
        agent_name: The name of the agent to hand off the conversation to. Required parameter.
        message: Message to the agent receiving the hand off. Include all the information required for the agent to complete the task that is handed off.  Required parameter.
        summary: Summary of the task that is handed off. Include any useful details for any work already done.  Required parameter.

        Returns
        response: The response from the agent.
        """
        handoff = HandOff(agent=agent_name, message=message, summary=summary)
        print("📦 delegate_task:", handoff)
        delegate_task_stack.append(handoff)
        if len(delegate_task_stack) > 1:
            return "Error: More than one hand off detected. Please perform only one hand off at a time. This hand off will be ignored."
        return handoff

    delegate_task.name = HAND_OFF_TOOL
    return delegate_task


class GroupAgent(ReActAgent):
    """
    We need to add few more utilities to the agent to allow handoff to other agents and streamline tool usage.
    """

    def __init__(self,
                 agent_llm: BaseChatModel,
                 name: str,
                 role_in_group: str,
                 persona: str,
                 task: Optional[str] = None,
                 tools: List[StructuredTool] = []):
        """
        Initialize the agent group.
        Parameters:
        ----------
        agent_llm : BaseChatModel
            The language model to be used by the agent.
        name : str
            The name of the agent.
        role_in_group : str
            The role of the agent in the group.
        persona : str
            The persona or role of the agent.
        task : Optional[str], optional
            The specific task assigned to the agent, by default None.
        tools : List[StructuredTool], optional
            A list of tools that the agent can use, by default an empty list.
        """
        super().__init__(agent_llm, name, persona, task, tools)
        self.role_in_group = role_in_group

    @observe(as_type="agent")
    def call_with_history(self, messages: list, message: Optional[str] = None) -> AgentReply:
        """
        Call the agent with the history of the conversation.
        Merges the history of the conversation with the current messages.
        Parameters:
        ----------
        messages : list
            The messages in the conversation.
        message : str, optional
            A new message to send to the agent, by default None.

        Returns:
        -------
        AgentReply
            The response from the agent.
        """
        langfuse_client.update_current_span(
            name=f"🤖💬 {self.name}",
            metadata={
                "agent": {
                    "agent_name": self.name,
                    "persona": self.persona,
                    "task": self.task,
                    "has_tools": len(self.tools) > 0,
                    "num_tools": len(self.tools)
                }
            }
        )
        # Merge the history of the conversation with the current messages by deduplicating messages by id
        agent_messages_by_content = {
            message.content: message for message in self.messages}
        new_messages = [
            message for message in messages if message.content not in agent_messages_by_content]
        self.messages += new_messages
        if message:
            self.messages.append(HumanMessage(message))
        response = self._call_llm_and_process_response()
        while self.is_last_message_tool_call() and not self.is_last_tool_call_handoff():
            response = self._call_with_tool_result()
        return response

    def initialize_in_group(self, other_agents: List['GroupAgent'], had_off_tool: StructuredTool):
        """
        Initialize the agent in the group.
        Parameters:
        ----------
        other_roles : List[str]
            The roles of the other agents in the group.
        """
        self.tools.append(had_off_tool)
        self.llm = self.llm.bind_tools(self.tools)

        # We need to override system message to provide more information
        other_roles = "\n- ".join(
            [f'**{a.name}**: {a.role_in_group}' for a in other_agents if a != self])
        self.messages = [SystemMessage(f"""{self.persona}

You are working in a group with other agents. Your role in the group is "{self.role_in_group}".
You need to follow your role and use available tools. If you can't solve the problem or need assistance you need to delegate_task task to the most appropriate agent, or ask the Manager for help
Other agents in your group are:
- {other_roles}

Use {HAND_OFF_TOOL} tool to hand off the task to another agent. Do not call tools as the names of agents. Delegate to one agent at a time! Do not do parallel delegations, this will result in an error.
Do your part of the task first, then delegate the rest to the agent who can complete the rest of the task. This is a group effort and we need to work together to solve the problem.
**Note** that tool name has to comply with following pattern `^[a-zA-Z0-9_-]+$`! Failure to comply with this will result in a fatal error.

{'Your task is:' + self.task if self.task else ''}""")]

    def is_last_tool_call_handoff(self):
        print("tool message:", self.messages[-1])
        return self.is_last_message_tool_call() and self.messages[-1].name and self.messages[-1].name == HAND_OFF_TOOL

Now we will build the agent group

In [50]:
def create_default_manager() -> GroupAgent:
    return GroupAgent(
        llm,
        "Group Manager",
        "To manage agent' communication and control overall flow",
        "You are experienced manager. "
        "You always make sure that your subordinates have all necessary information to solve their task."
        "You always find the best person for the given task",
        """1. You need to monitor communication of other agents
2. Hand off work to the agent that is most suited to solve it
3. Decide when the task is done and summarize the solution for the user
4. If you see that task can't be solved, ask for more clarifications from the user""")


class AgentGroup(object):
    """
    Manage multiple agents.
    We need to limit number of iterations the group or individual agent can take as agents might go into a loop and "eat" a lot of tokens.
    """

    def __init__(self,
                 agents: List[GroupAgent],
                 manager: GroupAgent = None,
                 max_delegations: int = 10,
                 ):
        """
        Initialize the agent group.
        Parameters:
        ----------
        agents : List[GroupAgent]
            The agents in the group.
        manager : GroupAgent, optional
            The manager of the group, by default None.
        """
        self.max_delegations = max_delegations
        self.delegate_task_stack = []
        self.delegate_task = create_handoff_tool(self.delegate_task_stack)
        self.agents = agents
        self.manager = manager if manager else create_default_manager()
        all_agents = self.agents + [self.manager]
        for agent in all_agents:
            agent.initialize_in_group(all_agents, self.delegate_task)
        self.messages = []

    def __repr__(self):
        all_agents = '\n - '.join([a.name for a in self.agents +
                                   [self.manager]])
        return f"AgentGroup with:\n{all_agents}"

    @staticmethod
    def __create_handoff_ai_message(active_agent: GroupAgent, delegate_task_message: HandOff) -> AIMessage:
        return AIMessage(f"""Agent '{active_agent.name}' handed off the task to agent '{delegate_task_message.agent}'
Summary of work done by  '{active_agent.name}':
```
{delegate_task_message.summary}
```
Reason for handoff to '{delegate_task_message.agent}':
```
{delegate_task_message.message}
```
""")

    def __get_agent_by_name(self, agent_name: str) -> Optional[GroupAgent]:
        for agent in (self.agents + [self.manager]):
            if agent.name == agent_name:
                return agent
        return None

    @observe(name="👫Agent Group Conversation", as_type='chain')
    def call(self, message: str):
        """
        Call the group of agents. This is effectivelly a state machine that manages the flow of the conversation.
        Here is how it works:
        1. We always start with the manager
        2. If there is a delegation, we hand off to the agent that was delegated the task
            1. If the agent is not found, we return an error and ask the agent to refer to the existing agents in the group
            2. We record the history of hand off in the group as it provides context for the other agents
        3. If the agent replied with a message, we hand off to the manager, as only the manager can reply to the user
        4. If the reply is not a tool call and the agent is the manager we return the response

        """
        print('🙃💬--> Calling agent group with message', message)
        self.messages.append(HumanMessage(message))
        # We always start with the manager
        active_agent = self.manager
        response = active_agent.call_with_history(self.messages)
        handoffs = 0
        while (active_agent.is_last_message_tool_call() or active_agent != self.manager) and handoffs < self.max_delegations:
            if len(self.delegate_task_stack) > 1:
                # I decided to not allow more than one hand off. You can change logic and allow parallel hand offs and execution
                self.delegate_task_stack.clear()
                response = active_agent.call(
                    "You can only hand off to one agent. Select the most appropriate agent and hand off to that agent.")
            elif len(self.delegate_task_stack) == 1:
                delegate_task_message = self.delegate_task_stack.pop()
                delegate_task_agent = self.__get_agent_by_name(delegate_task_message.agent)
                if not delegate_task_agent:
                    response = active_agent.call(f'Error: Agent {delegate_task_message.agent} not found, please refer to existing agents in the group.')
                else:
                    handoffs += 1
                    # We record the history of hand off in the group as it provides context for the other agents instead of passing all the messages every agent in the group had
                    ai_message = self.__create_handoff_ai_message(
                        active_agent, delegate_task_message)
                    self.messages.append(ai_message)
                    print(
                        f'\n🤝====Delegate "{active_agent.name}" ---> "{delegate_task_agent.name}"====\nDelegate message: "{delegate_task_message.message}"\n')
                    active_agent = delegate_task_agent
                    response = active_agent.call_with_history(self.messages, delegate_task_message.message)

            else:
                # This means that an agent that is not the manager replied with a message, we need to hand off to the manager
                last_message = AIMessage(
                    f'"{active_agent.name}": {response.reply}')
                self.messages.append(last_message)
                active_agent = self.manager
                response = active_agent.call_with_history(self.messages)
        print(f'🙃💬<-- Agent {active_agent.name} replied')
        print_markdown(response.reply)

## 🛫🏨 Simon Travel
Now we will build a travel agency that would be able to build complex trips for the customers.

We will need following agents:
1. **Flight researcher**. This agent will be responsible to research available flights for given destination and time.
2. **Activities Manager**. This agent will be planning activities for the customer.
3. **Lodging Expert**. This expert will be responsible for finding the best lodging for the clients.
4. **Accountant**. This agent will make sure that the travel package will not go over budget.
5. **Agency Manager** This is the group manager, but it has instructions specific for the travel agency

Travel agents will need tools to perform their tasks. We will define a few mock tools that will return some predefined results.

In [51]:
class Flight(BaseModel):
    destination: str
    origin: str
    price: float
    departure_date: str
    return_date: str
    flight_duration: int


@tool
def find_flights(origin: str, destination: str, departure_date: str, return_date: str) -> List[Flight]:
    """
    Find flights to the destination.
    
    Parameters
    origin: The city of origin.
    destination: The city.
    departure_date: The departure date.
    return_date: The return date.
    
    Returns
    flights: The flights.
    """
    print(f"🛫 find_flights: {origin} -> {destination} on {departure_date} -> {return_date}")
    return [
        Flight(destination=destination, origin=origin, price=200, departure_date=departure_date,
               return_date=return_date, flight_duration=10),
        Flight(destination=destination, origin=origin, price=100, departure_date=departure_date,
               return_date=return_date, flight_duration=15),
    ]


flight_researcher = GroupAgent(llm,
                               name='Flight Researcher',
                               role_in_group='Research available flights for the given destination and time',
                               persona='You are an expert in finding the best flights for the customers.',
                               task='Find the best flights for the customer.',
                               tools=[find_flights]
                               )


class Activity(BaseModel):
    name: str
    description: str
    price: float
    duration: int


@tool
def find_activities(destination: str, start_date: str, end_date: str) -> List[Activity]:
    """
    Find activities in the destination.
    
    Parameters
    destination: The destination.
    start_date: The start date.
    end_date: The end date.
    
    Returns
    activities: The activities.
    """
    print(f"🎭 find_activities: {destination} on {start_date} -> {end_date}")
    return [
        Activity(name="Visit the Eiffel Tower", description="Visit the Eiffel Tower in Paris.", price=50, duration=3),
        Activity(name="Louvre Museum", description="Visit the Louvre Museum in Paris.", price=30, duration=4),
    ]


activities_manager = GroupAgent(llm,
                                name='Activities Manager',
                                role_in_group='Plan activities for the customer',
                                persona='You are an expert in planning activities for the customers.',
                                task='Plan the best activities for the customer.',
                                tools=[find_activities]
                                )


class Lodging(BaseModel):
    name: str
    description: str
    price_per_night: float
    check_in_date: str
    check_out_date: str


@tool
def find_lodging(destination: str, check_in_date: str, check_out_date: str) -> List[Lodging]:
    """
    Find lodging in the destination.
    
    Parameters
    destination: The destination.
    check_in_date: The check-in date.
    check_out_date: The check-out date.
    
    Returns
    lodging: The lodging.
    """
    print(f"🏨find_lodging: {destination} on {check_in_date} -> {check_out_date}")
    return [
        Lodging(name="Hotel Paris", description="Hotel in Paris.", price_per_night=100, check_in_date=check_in_date,
                check_out_date=check_out_date),
        Lodging(name="Airbnb Paris", description="Airbnb in Paris.", price_per_night=50, check_in_date=check_in_date,
                check_out_date=check_out_date),
    ]


lodging_expert = GroupAgent(llm,
                            name='Lodging Expert',
                            role_in_group='Find the best lodging for the clients',
                            persona='You are an expert in finding the best lodging for the customers.',
                            task='Find the best lodging for the customer.',
                            tools=[find_lodging]
                            )


@tool
def calculate_tax(total: float) -> float:
    """
    Calculate the tax.
    
    Parameters
    total: Amount to tax.
    
    Returns
    tax: The tax.
    """
    print(f"🤓calculate_tax: {total}")
    return total * 0.1


import math
import numexpr as ne


@tool
def calculator(expression: str) -> str:
    """
    Calculate expression using Python's numexpr library.
    
    Parameters
    expression: The expression to calculate.
    
    Returns
    result: The result of the calculation.
    """
    print(f"🧮calculator: {expression}")
    local_dict = {"pi": math.pi, "e": math.e}
    return str(ne.evaluate(expression.strip(), local_dict=local_dict))


accountant = GroupAgent(llm,
                        name='Accountant',
                        role_in_group='Calculate all the costs and taxes',
                        persona='You are an expert in accounting.',
                        task='Make sure that the travel package will not go over budget.',
                        tools=[calculate_tax, calculator]
                        )


## Building the group
Now we will build the group of agents and the manager.

In [52]:
agency_manager = GroupAgent(
    llm,
    name='Agency Manager',
    role_in_group='Manage the travel agency',
    persona="You are experienced manager. "
            "You always make sure that your subordinates have all necessary information to solve their task. "
            "You always find the best person for the given task. "
            "You make your customers feel valued and appreciated. You are the face of the company.",
    task="""1. You need to monitor communication of other agents
2. Hand off work to the agent that is most suited to solve it. Do not do the work that more specialized agents can do.
3. Decide when the task is done and summarize the solution for the customer
4. If you see that task can't be solved, ask for more clarifications from the customer
5. Make sure that the package looks and sounds very appealing:
  a. Summarize the travel package for the client.
  b. Add colorful description of the trip, make it sound very appealing.
  c. Use markdown format in the response so it looks very pretty and visually appealing.
6. Do not ask followup questions, make safe assumptions or consider different options.
"""
)

travel_agents = [flight_researcher,
                 activities_manager, lodging_expert, accountant]

travel_group = AgentGroup(travel_agents, agency_manager)
print(activities_manager.messages[0].content)

travel_group

You are an expert in planning activities for the customers.

You are working in a group with other agents. Your role in the group is "Plan activities for the customer".
You need to follow your role and use available tools. If you can't solve the problem or need assistance you need to delegate_task task to the most appropriate agent, or ask the Manager for help
Other agents in your group are:
- **Flight Researcher**: Research available flights for the given destination and time
- **Lodging Expert**: Find the best lodging for the clients
- **Accountant**: Calculate all the costs and taxes
- **Agency Manager**: Manage the travel agency

Use delegate_task tool to hand off the task to another agent. Do not call tools as the names of agents. Delegate to one agent at a time! Do not do parallel delegations, this will result in an error.
Do your part of the task first, then delegate the rest to the agent who can complete the rest of the task. This is a group effort and we need to work together 

AgentGroup with:
Flight Researcher
 - Activities Manager
 - Lodging Expert
 - Accountant
 - Agency Manager

In [53]:
travel_group.call(
    'Build a trip to Paris from 2024-11-13 to 2024-11-20. We are flying from Munich Germany. My wife and I want to see a few sights and stay in a nice hotel. Our budget is 5000 euros.')

🙃💬--> Calling agent group with message Build a trip to Paris from 2024-11-13 to 2024-11-20. We are flying from Munich Germany. My wife and I want to see a few sights and stay in a nice hotel. Our budget is 5000 euros.
🔧---> Invoking tool call {'name': 'delegate_task', 'args': {'agent_name': 'Flight Researcher', 'message': 'Please research available flights from Munich, Germany to Paris for two people, departing on 2024-11-13 and returning on 2024-11-20. The budget for the entire trip is 5000 euros, so keep that in mind when selecting flights.', 'summary': 'Customer wants a trip to Paris from 2024-11-13 to 2024-11-20, flying from Munich, Germany. Two people traveling, budget is 5000 euros. They want to see a few sights and stay in a nice hotel.'}, 'id': 'call_ZnKWlR82hLdsNAvkGlcUNgEY', 'type': 'tool_call'}
📦 delegate_task: agent='Flight Researcher' message='Please research available flights from Munich, Germany to Paris for two people, departing on 2024-11-13 and returning on 2024-11-20

Here is a wonderful travel package for your trip to Paris from November 13 to November 20, 2024:

### Flight Options:
- Option 1: Flights from Munich to Paris at 200 euros per person (10 hours duration)
- Option 2: Flights from Munich to Paris at 100 euros per person (15 hours duration)

### Lodging Options:
- Hotel Paris: A charming hotel stay at 100 euros per night
- Airbnb Paris: A cozy Airbnb option at 50 euros per night

### Activities:
- Visit the iconic Eiffel Tower (50 euros per person)
- Explore the world-famous Louvre Museum (30 euros per person)

### Cost Summary:
- Option 1 (Hotel + Flight 200 euros): Total cost approximately 2156 euros (including taxes)
- Option 2 (Airbnb + Flight 100 euros): Total cost approximately 1166 euros (including taxes)

---

### Why this trip is perfect for you:
Imagine strolling hand-in-hand with your loved one through the romantic streets of Paris, gazing up at the magnificent Eiffel Tower sparkling against the night sky. Enjoy the rich culture and history as you wander through the Louvre, marveling at timeless masterpieces. After a day of adventure, retreat to your comfortable and elegant lodging, whether a charming hotel or a cozy Airbnb, tailored to your preference and budget.

This carefully curated trip offers the perfect blend of iconic sights, comfort, and affordability, all within your budget of 5000 euros. It's a dream getaway designed to create unforgettable memories for you and your wife.

Would you like me to assist with booking or provide any additional details?

# What is next?
1. **Learn**. I recommend taking a few courses from [Deeplearning.AI](https://www.deeplearning.ai/courses/?courses_date_desc%5BrefinementList%5D%5Bcourse_type%5D%5B0%5D=Short%20Courses). There are plenty of free and short (under a day) courses there for you to take.
2. **Explore** real agentic frameworks:
    1. [Pydantic AI](https://ai.pydantic.dev/) is a straightforward and straight forward framework I personally used in multiple production projects. It is straightforward to use and gives you a lot of control.
    1. [LangGraph](https://www.langchain.com/langgraph) is [LangChain's](https://www.langchain.com/) take on agentic workflows. Be aware it is not the most straight forward one out there, but it gives you a lot of control.
    2. [CrewAI](https://www.crewai.com/) It is a communication-based agentic framework. We built a similar framework in this talk. The abstractions are clear and give you a good mental model to work with. I would not use it in production for anything complex as my (limited) testing showed that it might be hard to control. Excellent for prototyping.
    3. [AutoGen](https://microsoft.github.io/autogen/0.2/) is a child of a [research paper](https://arxiv.org/abs/2308.08155). It is very powerful, but being the child of academia is not the best thing to work with, at least in its early incarnations. It got a lot of attention and funding (Microsoft), so I am hopeful it got much better. I have used it on several occasions when I wanted to prototype something quickly in two articles I have written [Think Tank: Automated AI Expert Group](https://www.ai-engineer.me/think-tank-automated-ai-expert-group/) and [Coding Interviews and AI](https://www.ai-engineer.me/coding-interviews-and-ai-are-traditional-methods-still-relevant/)
3. **Stay connected (Shameless Plug)** subscribe to my blog [AI Engineer](https://www.ai-engineer.me/) and/or stay in touch via [LinkedIn](https://www.linkedin.com/in/oleksandr-antoshchenko/)